# 02 · REAL / FAKE news classifier

Fine-tune `symanto/xlm-roberta-base-snli-mnli-anli-xnli` (a 3-way NLI checkpoint)
on the [Fake or Real News dataset](https://www.kaggle.com/datasets/rajatkumar30/fake-news)
as a **binary REAL / FAKE classifier** (the 3-class head is replaced by
`num_labels=2` + `ignore_mismatched_sizes=True`), then run inference on a transcript
produced by notebook 01.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if torch.cuda.is_available():
    print(f"gpu:    {torch.cuda.get_device_name(0)}")

## 2. Fine-tune

In [ ]:
from src.train import TrainConfig, train

cfg = TrainConfig(
    epochs=3,
    batch_size=8,
    learning_rate=2e-5,
    output_dir=ROOT / "output",
    seed=42,
)
trainer = train(cfg)

## 3. Classify a transcript

Use the `.txt` from notebook 01 (or the included `data/sample_input.txt`).

In [ ]:
from src.predict import classify_text

txt_path = ROOT / "data" / "sample_input.txt"
label = classify_text(
    txt_path.read_text(encoding="utf-8"),
    model_dir=cfg.output_dir / "final",
)
print(f"{txt_path.name}: {label}")